In [1]:
!git clone https://github.com/swetha3456/TiSASRec.pytorch.git
%cd TiSASRec.pytorch

Cloning into 'TiSASRec.pytorch'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 106 (delta 64), reused 59 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 6.89 MiB | 16.06 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/kaggle/working/TiSASRec.pytorch


In [2]:
import json
import pandas as pd
from datetime import datetime
from tqdm import tqdm

def process_jsonl_data(input_path, output_path):
    processed_rows = []
    
    print(f"Reading JSONL from {input_path}...")
    with open(input_path, 'r') as f:
        for line in tqdm(f):
            # 1. Parse each line as an individual JSON object
            entry = json.loads(line)
            uid = entry['user_id']
            
            # 2. Extract History (fast_memory_ids)
            history_ids = entry['fast_memory_ids']
            history_ts = entry['fast_id_timestamps']
            
            for item, ts in zip(history_ids, history_ts):
                dt_obj = datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
                processed_rows.append([uid, item, int(dt_obj.timestamp())])
                
            # 3. Extract the Target Item
            target_item = entry['target_item']
            target_ts = entry['target_timestamp']
            target_dt = datetime.strptime(target_ts, '%Y-%m-%d %H:%M:%S')
            processed_rows.append([uid, target_item, int(target_dt.timestamp())])

    # Create DataFrame
    df = pd.DataFrame(processed_rows, columns=['user_id', 'item_id', 'timestamp'])
    
    # Sort chronologically (Crucial for sequential models)
    df = df.sort_values(by=['user_id', 'timestamp'])

    # 4. Map IDs to 1-indexed integers
    print("Mapping IDs to integers...")
    user_map = {val: i + 1 for i, val in enumerate(df['user_id'].unique())}
    item_map = {val: i + 1 for i, val in enumerate(df['item_id'].unique())}
    
    df['user_id'] = df['user_id'].map(user_map)
    df['item_id'] = df['item_id'].map(item_map)

    # 5. Save as Tab-Separated (no header)
    df.to_csv(output_path, sep='\t', index=False, header=False)
    
    print(f"--- Done ---")
    print(f"Saved to: {output_path}")
    return item_map

item_mapping = process_jsonl_data('/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/e-commerce_sequences.jsonl', 'data/ecommerce_dataset.txt')

Reading JSONL from /kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/e-commerce_sequences.jsonl...


272074it [01:17, 3492.19it/s]


Mapping IDs to integers...
--- Done ---
Saved to: data/ecommerce_dataset.txt


In [3]:
import pandas as pd

# Load your processed file
file_path = 'data/ecommerce_dataset.txt' # Change to your actual path
df = pd.read_csv(file_path, sep='\t', header=None, names=['uid', 'iid', 'ts'])

# 1. Force items to be 1-indexed integers
unique_items = df['iid'].unique()
item_map = {old_id: i + 1 for i, old_id in enumerate(sorted(unique_items))}
df['iid'] = df['iid'].map(item_map)

# 2. Force users to be 1-indexed integers
unique_users = df['uid'].unique()
user_map = {old_id: i + 1 for i, old_id in enumerate(sorted(unique_users))}
df['uid'] = df['uid'].map(user_map)

# 3. Save it back
df.to_csv(file_path, sep='\t', index=False, header=False)

# 4. CRITICAL: Calculate the parameters you will pass to the script
new_itemnum = df['iid'].max()
new_usernum = df['uid'].max()

print(f"Verified itemnum: {new_itemnum}")
print(f"Verified usernum: {new_usernum}")

Verified itemnum: 32529
Verified usernum: 5511


In [4]:
# Load original category mapping (original_id -> category)
with open("/kaggle/input/datasets/dyuthivivek/ecommerce-recsys-sequences/category_mappings.json", "r") as f:
    old_mapping = json.load(f)["product_to_category"]

# Create the new mapping using the new integer IDs
new_product_to_category = {}
for original_id, cat_id in old_mapping.items():
    # Only map items that actually exist in your new dataset
    if str(original_id) in item_mapping:
        new_id = item_mapping[str(original_id)]
        new_product_to_category[new_id] = cat_id

# Save this for TiSASRec main.py to use
with open("data/category_mapping.json", "w") as f:
    json.dump({"product_to_category": new_product_to_category}, f)

In [5]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [6]:
all_mapped_items = set(df['iid'].unique())
mapped_items_with_cats = set(new_product_to_category.keys())

missing = all_mapped_items - mapped_items_with_cats
if missing:
    print(f"WARNING: {len(missing)} items in your data have NO category! Example: {list(missing)[:5]}")

In [7]:
!python main.py --dataset=ecommerce_dataset --train_dir=default --device=cuda --lr=5e-4 --hidden_units=8 --dropout_rate=0.5 --num_epochs=100

Loading category mappings for hard negative sampling...
Preparing data...
Preparing done...
average sequence length: 787.15
Preparing relation matrix: 100%|███████████| 5511/5511 [00:07<00:00, 696.02it/s]
loss in epoch 1 iteration 0: 1.381162166595459
loss in epoch 1 iteration 10: 1.378892183303833
loss in epoch 1 iteration 20: 1.3757712841033936
loss in epoch 1 iteration 30: 1.3730406761169434
loss in epoch 1 iteration 40: 1.3682312965393066
loss in epoch 2 iteration 0: 1.3669136762619019
loss in epoch 2 iteration 10: 1.3601295948028564
loss in epoch 2 iteration 20: 1.3569320440292358
loss in epoch 2 iteration 30: 1.3537492752075195
loss in epoch 2 iteration 40: 1.3446028232574463
loss in epoch 3 iteration 0: 1.344036340713501
loss in epoch 3 iteration 10: 1.3305240869522095
loss in epoch 3 iteration 20: 1.3286820650100708
loss in epoch 3 iteration 30: 1.3173766136169434
loss in epoch 3 iteration 40: 1.3154538869857788
loss in epoch 4 iteration 0: 1.3042023181915283
loss in epoch 4 it